# Experiment 4: GRPO Reinforcement Learning on Qwen2.5-1.5B (TPU)

This notebook performs **Group Relative Policy Optimization (GRPO)** to align the SFT-trained model with mode-aware rewards. It requires a checkpoint from Experiment 1 or 3.

### 1. Environment Setup

In [ ]:
%%capture
!pip install "google-tunix[prod]"
!pip install wandb huggingface_hub gcsfs datasets evaluate tqdm jsonlines python-dotenv

### 2. Verify TPU Access

In [ ]:
import jax
print("TPU devices:", jax.devices())

### 3. Clone Repository & Setup Rewards

In [ ]:
import os
REPO_URL = "https://github.com/TinevimboMusingadi/scot-reasoning-.git"
REPO_DIR = "scot-reasoning-"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL}

import sys
sys.path.append(os.path.join(os.getcwd(), REPO_DIR, "training"))
from rewards import total_reward

### 4. Initialize RL Cluster (Tunix)

In [ ]:
import jax.numpy as jnp
import optax
from tunix.rl import grpo_learner, rl_cluster
from tunix import GRPOConfig, ClusterConfig, RLTrainingConfig, RolloutConfig
from tunix.models.qwen2 import model as qwen_lib
from tunix.models.qwen2 import params_safetensors as qwen_params
from transformers import AutoTokenizer

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
MESH = [(1, 8), ("fsdp", "tp")]
mesh = jax.make_mesh(*MESH, axis_types=(jax.sharding.AxisType.Auto,) * 2)
SFT_CKPT = "/content/drive/MyDrive/scot-results/exp1-qwen-sft/"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
SCOT_TOKENS = ["<reasoning>", "</reasoning>", "<meta_reasoning>", "</meta_reasoning>", 
               "<abduction>", "</abduction>", "<decompose>", "</decompose>", 
               "<deduction>", "</deduction>", "<induction>", "</induction>", 
               "<analogy>", "</analogy>", "<causal>", "</causal>", "<answer>", "</answer>"]
tokenizer.add_special_tokens({"additional_special_tokens": SCOT_TOKENS})

config = qwen_lib.ModelConfig.qwen2_5_1_5b()
config.vocab_size = len(tokenizer)

c_config = ClusterConfig(
    role_to_mesh={rl_cluster.Role.ACTOR: mesh, rl_cluster.Role.REFERENCE: mesh},
    training_config=RLTrainingConfig(
        eval_every_n_steps=50,
        max_steps=300,
        actor_optimizer=optax.adamw(learning_rate=5e-6),
    ),
    rollout_config=RolloutConfig(
        max_tokens_to_generate=512,
        temperature=0.9,
    )
)

def model_factory(mesh):
    with mesh:
        # In real use, point this to your SFT checkpoint
        # return qwen_params.create_model_from_safe_tensors(SFT_CKPT, config, mesh)
        return qwen_params.create_model_from_safe_tensors(MODEL_ID, config, mesh)

cluster = rl_cluster.RLCluster(
    actor=model_factory(mesh),
    reference=model_factory(mesh),
    tokenizer=tokenizer,
    cluster_config=c_config,
)

### 5. Training with GRPOLearner

In [ ]:
def reward_fn(completions, ground_truths):
    return [total_reward(c, gt) for c, gt in zip(completions, ground_truths)]

algo_config = GRPOConfig(num_generations=8, beta=0.04)
learner = grpo_learner.GRPOLearner(rl_cluster=cluster, algo_config=algo_config, reward_fns=[reward_fn])

import jsonlines
DATA_PATH = os.path.join(REPO_DIR, "data/full_run/scot_traces.jsonl")
train_data = []
with jsonlines.open(DATA_PATH) as reader:
    for row in reader:
        train_data.append({"prompt": f"<|im_start|>user\n{row['problem']}<|im_end|>\n<|im_start|>assistant\n", 
                           "ground_truth": row["answer"]})

learner.train(train_ds=train_data[:500]) # Example subset